In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim

c:\Users\Bhavya sri\anaconda3\envs\hatespeechenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [2]:
df = pd.read_csv('labeled_data.csv')
print(df.columns)
df.head()


Index(['Unnamed: 0', 'count', 'hate_speech', 'offensive_language', 'neither',
       'class', 'tweet'],
      dtype='object')


,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [3]:
df = pd.read_csv('labeled_data.csv')  
print(df)       # Or your actual file name
df = df[['tweet', 'class']].dropna()         # Keep only tweet + class
print(df)
df['class'] = df['class'].apply(lambda x: 1 if x != 2 else 0)
print(df)
df = df[df['class'].isin([0, 1])]            # Filter only 0 and 1
df = df.reset_index(drop=True)               # Reset index after filtering
df.columns = ['text', 'label']               # Rename for consistency
df['label'] = df['label'].astype(int)
print("Unique labels:", df['label'].unique())  # Double check labels


       Unnamed: 0  count  hate_speech  offensive_language  neither  class  \
0               0      3            0                   0        3      2   
1               1      3            0                   3        0      1   
2               2      3            0                   3        0      1   
3               3      3            0                   2        1      1   
4               4      6            0                   6        0      1   
...           ...    ...          ...                 ...      ...    ...   
24778       25291      3            0                   2        1      1   
24779       25292      3            0                   1        2      2   
24780       25294      3            0                   3        0      1   
24781       25295      6            0                   6        0      1   
24782       25296      3            0                   0        3      2   

                                                   tweet  
0      !!! RT @m

In [4]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)


In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class AbuseDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=128, return_tensors='pt'
        )
        self.labels = torch.tensor(labels.values)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = AbuseDataset(train_texts, train_labels)
val_dataset = AbuseDataset(val_texts, val_labels)

In [6]:
class BERTClassifier(nn.Module):
    def __init__(self):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        return self.classifier(output)


In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)



In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)


In [9]:
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device).long()

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


Epoch 1/3, Loss: 163.8230
Epoch 2/3, Loss: 90.2232
Epoch 3/3, Loss: 54.6900


In [15]:
from sklearn.metrics import accuracy_score,f1_score,precision_score, recall_score

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1_score=f1_score(all_labels, all_preds, average='weighted')
precision_score= precision_score(all_labels, all_preds)
recall_score = recall_score(all_labels, all_preds)


print(f"Validation Accuracy: {acc:.2%}")
print(f"Validation F1 Score: {f1_score:.2%}")
print(f"Validation Recall Score: {recall_score:.2%}")
print(f"Validation precision Score: {precision_score:.2%}")


Validation Accuracy: 95.70%
Validation F1 Score: 95.61%
Validation Recall Score: 98.42%
Validation precision Score: 96.48%


In [16]:
def predict_text(text, model, tokenizer, device):
    model.eval()
    with torch.no_grad():
        encoding = tokenizer(
            text, truncation=True, padding='max_length', max_length=128, return_tensors='pt'
        )
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask)
        pred = torch.argmax(outputs, dim=1).item()
    return pred


sample_text = "i hate you"
result = predict_text(sample_text, model, tokenizer, device)
print(f"Text: '{sample_text}' → {result}")

sample_text = "you are beautiful"
result = predict_text(sample_text, model, tokenizer, device)
print(f"Text: '{sample_text}' → {result}")


Text: 'i hate you' → 1
Text: 'you are beautiful' → 0


In [17]:
import os

# Create a folder named "model" if it doesn't exist
os.makedirs("model", exist_ok=True)

# Save the trained model weights
torch.save(model.state_dict(), "model/bert_model.pt")
print("Model saved successfully!")


Model saved successfully!
